In [5]:
import pandas as pd
df = pd.read_csv('sales_sample.csv')
print(df.head())

import datetime as dt

# Convertir la columna Date a formato fecha
df['Date'] = pd.to_datetime(df['Date'])

# Fecha de referencia = un día después de la última venta
fecha_ref = df['Date'].max() + dt.timedelta(days=1)
print(f"Fecha de referencia: {fecha_ref}")

# Calcular RFM por cliente
rfm = df.groupby('CustomerID').agg({
    'Date': lambda x: (fecha_ref - x.max()).days,  # Recencia
    'CustomerID': 'count',                          # Frecuencia
    'Revenue': 'sum'                                # Monetario
})

# Renombrar columnas
rfm.columns = ['Recencia', 'Frecuencia', 'Monetario']

# Mostrar las primeras 10 filas
print(rfm.head(10))
print(f"\nTotal de clientes: {len(rfm)}")


         Date   Region    Product       Category SalesRep  Quantity  \
0  2024-11-17     West  Product D  Home & Garden    Alice        67   
1  2023-04-13  Central  Product C          Books    Diana        30   
2  2023-03-29     East  Product B       Clothing      Bob        76   
3  2023-09-13  Central  Product A          Books  Charlie        29   
4  2023-10-11  Central  Product E       Clothing      Bob         3   

   UnitPrice  Discount    Revenue  CustomerID  
0      28.61      0.01   2467.010        7247  
1     497.35      0.11  13279.240        9432  
2     246.17      0.01  18521.830        2896  
3     380.44      0.03  10701.780        8883  
4     231.63      0.18    740.753        4324  
Fecha de referencia: 2025-01-01 00:00:00
            Recencia  Frecuencia  Monetario
CustomerID                                 
1000             276           1  24956.470
1005             247           1  12349.560
1012             117           1   6882.250
1016             216    

In [6]:
# Crear cuartiles (1-4) para cada métrica
rfm['R'] = pd.qcut(rfm['Recencia'], 4, labels=[4, 3, 2, 1])
rfm['F'] = pd.qcut(rfm['Frecuencia'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M'] = pd.qcut(rfm['Monetario'], 4, labels=[1, 2, 3, 4])

# Crear el segmento combinado
rfm['Segmento'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)

# Definir nombres de segmentos de negocio
def nombre_segmento(row):
    if row['R'] == 4 and row['F'] >= 3 and row['M'] >= 3:
        return 'VIP'
    elif row['R'] <= 2 and row['F'] >= 3 and row['M'] >= 3:
        return 'En riesgo'
    elif row['R'] == 4 and row['F'] == 1:
        return 'Nuevo'
    elif row['R'] <= 2 and row['F'] <= 2:
        return 'Perdido'
    else:
        return 'Regular'

rfm['Nombre_Segmento'] = rfm.apply(nombre_segmento, axis=1)

# Ver cuántos clientes hay en cada segmento
print(rfm['Nombre_Segmento'].value_counts())

Nombre_Segmento
Regular      880
Perdido      460
En riesgo    210
VIP          150
Nuevo        102
Name: count, dtype: int64


In [7]:
# Exportar el resultado a CSV
rfm.to_csv('rfm_segmentos.csv')
print("Archivo exportado exitosamente")
print(f"Clientes totales: {len(rfm)}")
print(rfm.head())

Archivo exportado exitosamente
Clientes totales: 1802
            Recencia  Frecuencia  Monetario  R  F  M Segmento Nombre_Segmento
CustomerID                                                                   
1000             276           1   24956.47  3  1  4      314         Regular
1005             247           1   12349.56  3  1  3      313         Regular
1012             117           1    6882.25  4  1  2      412           Nuevo
1016             216           1   23358.77  3  1  4      314         Regular
1019             331           1   31958.39  3  1  4      314         Regular
